# DiMuonTagger quickstart (coffea-casa)

Runs `DiMuonProcessor` over the custom NanoAOD, the same as `analysis/scripts/run.py`, then shows the cutflow, the muon-ancestor table and a few shape plots.

Before you start:
1. Clone the repo into your coffea-casa home directory and open this notebook from `analysis/notebooks/`.
2. Make the fileset on lxplus with `analysis/scripts/build_fileset.py --tag <tag>`, then commit it (or copy it to `analysis/filesets/<tag>.json`).
3. Use the image's coffea. Don't `pip install --user` another coffea: the Dask workers run the image's version, and a different one in this pod can break pickling between the two. The cell below prints both.

In [ ]:
import os
import sys

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))   # this notebook lives in analysis/notebooks
sys.path.insert(0, os.path.join(REPO, "analysis"))

import awkward as ak
import coffea

from dimuontagger import plotting, runner
from dimuontagger.config import load_config
from dimuontagger.processor import DiMuonProcessor

print("coffea", coffea.__version__, "| awkward", ak.__version__, "| repo", REPO)

## Settings

Start small (`MAXCHUNKS = 1`) and check the ancestor table before a full run. Skims stay off until you set `SKIM_DEST`. The Dask workers write the skims, so the destination must be writable from the workers; your home directory here is not. Run the probe cell below before a full run.

In [ ]:
TAG = "v1"
FILESET = os.path.join(REPO, "analysis", "filesets", "%s.json" % TAG)
EXECUTOR = "dask"                    # "dask" (coffea-casa workers), "futures" or "iterative" (this pod)
SCHEDULER = "tls://localhost:8786"   # coffea-casa's Dask scheduler
CHUNKSIZE = 100_000
MAXCHUNKS = 1                        # per dataset; None = everything
MAX_FILES = None                     # per dataset; None = all files
PROCESSES = None                     # e.g. ["hahm_zd", "qcd_muenriched"]
ERAS = None                          # e.g. ["2022postEE"]
SKIM_DEST = None                     # e.g. "root://<writable xrootd endpoint>//store/user/<you>/DiMuonSkims/v1"
SKIM_STORAGE_OPTIONS = {}
OUTPUT = runner.default_output_path(TAG)

In [ ]:
fileset = runner.load_fileset(FILESET, processes=PROCESSES, eras=ERAS, max_files=MAX_FILES)
for name, entry in sorted(fileset.items()):
    meta = entry["metadata"]
    print("%-11s %-13s %5d files  %s" % (meta.get("role", ""), meta.get("era", ""), len(entry["files"]), name))

## Executor

For Dask, the `dimuontagger` package is zipped and uploaded to the workers (`runner.ship_package_to_workers`). The workers cannot see this repo, and workers that join later get the zip as well.

In [ ]:
client = None
if EXECUTOR == "dask":
    from dask.distributed import Client

    client = Client(SCHEDULER)
    print(client)
    print("coffea on workers:", set(client.run(lambda: __import__("coffea").__version__).values()))
    executor = runner.make_executor("dask", client=client)
else:
    executor = runner.make_executor(EXECUTOR, workers=4)

## Can the workers write the skims?

This writes, reads back and deletes a small parquet file under `SKIM_DEST`, on a worker when using Dask. If it fails, choose another destination, or run the skims with `EXECUTOR = "futures"` in this pod and a local `SKIM_DEST`.

In [ ]:
if SKIM_DEST:
    host, rows = runner.probe_destination(SKIM_DEST, SKIM_STORAGE_OPTIONS, client=client)
    print("wrote and read back %d rows under %s from %s" % (rows, SKIM_DEST, host))
else:
    print("SKIM_DEST not set: no skims")

## Run

In [ ]:
config = load_config()
proc = DiMuonProcessor(config, skim_dest=SKIM_DEST, skim_storage_options=SKIM_STORAGE_OPTIONS)
result = runner.run(fileset, proc, executor, chunksize=CHUNKSIZE, maxchunks=MAXCHUNKS)
record = runner.result_record(result, fileset, config, tag=TAG, skim_dest=SKIM_DEST)
runner.save(record, OUTPUT)
print("saved", OUTPUT)

## Cutflow and truth categories

In [ ]:
import pandas as pd

outdir = os.path.dirname(OUTPUT)
pd.DataFrame(plotting.write_cutflow(record, outdir)).set_index("dataset")

## Muon ancestors: check the LLP pdgIds

`truth.llp_pdgids` in `analysis/config.yaml` are unverified guesses. Signal datasets should show the LLP among the ancestors of most gen-matched muons. If one doesn't, fix the pdgId, or check whether NanoAOD's GenPart pruning dropped the LLP.

In [ ]:
table, missing = plotting.write_ancestors(record, outdir)
for name, info in table.items():
    top = ", ".join("%d (%.0f%%)%s" % (r["pdgid"], 100 * r["fraction"], "*" if r["configured_llp"] else "")
                    for r in info["rows"][:8])
    print("%-11s %s\n    %s" % (info["role"], name, top))
print("\n* = configured LLP pdgId")
if missing:
    print("WARNING: no configured LLP among the muon ancestors of", missing)

## Shapes per truth category

The full set (every variable, per era and for all eras) is written by `analysis/scripts/plot.py`, or by `plotting.all_plots(record, ...)`.

In [ ]:
from IPython.display import Image, display

datasets = sorted(record["output"])
for var in ("mass", "lxy", "lxy_sig", "pair_type"):
    path = plotting.shape_plot(record, var, datasets, "all eras", os.path.join(outdir, "plots", "all", var + ".png"))
    if path:
        display(Image(filename=path, width=520))

## Read the skims back

One parquet file per chunk under `SKIM_DEST/<process>/<era>/<primary>/`.

In [ ]:
files = sorted(set().union(*(out["skim_files"] for out in record["output"].values())))
print(len(files), "skim files")
if files:
    skim = ak.from_parquet(files[0], storage_options=SKIM_STORAGE_OPTIONS)
    print(len(skim), "candidates in", files[0])
    print(dict(zip(*__import__("numpy").unique(ak.to_numpy(skim.label), return_counts=True))))